# DPR Stage-1 run (top-5,000, 97 queries)

Produces `runs/dpr.parquet` and `runs/dpr_manifest.json` and copies them to Drive.
A **CPU runtime is enough**; a GPU only makes query encoding faster. About 15–20 minutes,
mostly downloading the 13.5 GB index from Drive.

## 1. Code and data

In [ ]:
BRANCH = 'shubham/dpr-run'
!rm -rf /content/IRE_PROJECT
!git clone -q -b {BRANCH} https://github.com/a-n-u-p-a-m/IRE_PROJECT.git /content/IRE_PROJECT
%cd /content/IRE_PROJECT
!git log --oneline -1
!pip install -q onnxruntime
!python -m experiments_shubham.fetch_trec_dl

## 2. Download the index through the Drive API

Not through the Drive mount: it drops out (`Errno 107`) on long reads of the shards.
Resumable: rerun the cell if it stops.

In [ ]:
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io, os

IRE_FOLDER_ID = '18aZ7hG6GRS0YdeAOb99rV8Nh8hET4Xcv'   # Drive folder IRE_PROJECT
svc = build('drive', 'v3')

def children(parent, extra=''):
    files, token = [], None
    while True:
        r = svc.files().list(q=f"'{parent}' in parents and trashed = false {extra}", pageSize=1000,
                             fields='nextPageToken, files(id, name, size)', pageToken=token).execute()
        files += r['files']; token = r.get('nextPageToken')
        if not token: return files

index = [f for f in children(IRE_FOLDER_ID, "and mimeType = 'application/vnd.google-apps.folder'")
         if f['name'] == 'dpr_index_cosdpr_distil']
assert len(index) == 1, index
files = children(index[0]['id'])
print(len(files), 'files on Drive')            # expect 179

DST = '/content/dpr_index'; os.makedirs(DST, exist_ok=True)
for i, f in enumerate(sorted(files, key=lambda f: f['name']), 1):
    out = os.path.join(DST, f['name'])
    if os.path.exists(out) and os.path.getsize(out) == int(f['size']):
        continue
    with io.FileIO(out + '.part', 'wb') as fh:
        dl = MediaIoBaseDownload(fh, svc.files().get_media(fileId=f['id']), chunksize=64 * 1024 * 1024)
        done = False
        while not done:
            _, done = dl.next_chunk(num_retries=5)
    os.replace(out + '.part', out)
    if i % 20 == 0: print(f'{i}/{len(files)}')
print('emb shards locally:', sum(n.startswith('emb_') for n in os.listdir(DST)))   # expect 89

## 3. Encoder check

Our encoder against Castorini's reference ONNX export on all 97 queries. Must print
min cosine > 0.9999 before the run is trusted.

In [ ]:
import numpy as np, onnxruntime as ort
from huggingface_hub import hf_hub_download
from experiments_shubham.dpr import CosDPREncoder
from experiments_shubham.run_dpr import read_queries

enc = CosDPREncoder(device='cpu')
texts = list(read_queries('data/queries.tsv').values())
sess = ort.InferenceSession(hf_hub_download('castorini/cosdpr-distil-onnx', 'cosdpr-distil-optimized.onnx'))
ref = np.stack([sess.run(['pooler_output'], {'input_ids': np.array([enc.tokenizer(t)['input_ids']], dtype=np.int64)})[0][0]
                for t in texts])
ref /= np.linalg.norm(ref, axis=1, keepdims=True)
cos = (enc.encode(texts) * ref).sum(1)
print(f'min cosine = {cos.min():.6f}')
assert cos.min() > 0.9999

## 4. Run (exact search over all 89 shards)

In [ ]:
!python -m experiments_shubham.run_dpr --index /content/dpr_index --device cpu

## 5. Sanity check against our earlier verified numbers

Expect DL19 nDCG@10 0.7063 / R@1000 0.8224 and DL20 0.6889 / 0.8587 (official linear gains).

In [ ]:
import gzip, urllib.request
import numpy as np, pandas as pd

run = pd.read_parquet('runs/dpr.parquet')
by_q = {q: g.sort_values('rank')['doc_id'].tolist() for q, g in run.groupby('query_id')}
qrels = {}
for line in open('data/qrels.txt'):
    q, _, d, r = line.split()
    qrels.setdefault(q, {})[d] = int(r)

def ndcg10(q):
    disc = lambda xs: sum(x / np.log2(i + 2) for i, x in enumerate(xs))
    gains = [qrels[q].get(d, 0) for d in by_q[q][:10]]
    return disc(gains) / disc(sorted(qrels[q].values(), reverse=True)[:10])

def recall1000(q):
    rel = {d for d, r in qrels[q].items() if r >= 2}
    return len(rel & set(by_q[q][:1000])) / len(rel) if rel else np.nan

BASE = 'https://msmarco.z22.web.core.windows.net/msmarcoranking/'
for year, f in [('dl19', 'msmarco-test2019-queries.tsv.gz'), ('dl20', 'msmarco-test2020-queries.tsv.gz')]:
    raw = gzip.decompress(urllib.request.urlopen(BASE + f).read()).decode()
    ids = {l.split('\t')[0] for l in raw.splitlines()} & set(by_q)
    print(f'{year}: {len(ids)} queries | nDCG@10 {np.mean([ndcg10(q) for q in ids]):.4f} '
          f'| R@1000 {np.nanmean([recall1000(q) for q in ids]):.4f}')
print('rows:', len(run), '| rows per query:', sorted(run.groupby('query_id').size().unique()))

## 6. Save the outputs to Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/IRE_PROJECT/runs
!cp runs/dpr.parquet runs/dpr_manifest.json /content/drive/MyDrive/IRE_PROJECT/runs/
!ls -la /content/drive/MyDrive/IRE_PROJECT/runs/